# Day 42: Custom Elementwise CUDA Kernel via Triton

**Layer:** Implementation


## Concept Overview

Triton is a Python DSL for writing GPU kernels at the tile level. A Triton kernel specifies parallelism via program_id and tl.arange, and the compiler handles thread binding, vectorization, and memory coalescing.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Writing a Custom Triton Kernel

Triton is a Python DSL that compiles to efficient GPU kernels. A Triton kernel is written as a Python function decorated with @triton.jit, with explicit tile-level parallelism.


In [ ]:
try:
    import triton
    import triton.language as tl

    @triton.jit
    def elementwise_silu_kernel(x_ptr, y_ptr, n, BLOCK: tl.constexpr):
        pid = tl.program_id(0)
        offs = pid * BLOCK + tl.arange(0, BLOCK)
        mask = offs < n
        x = tl.load(x_ptr + offs, mask=mask)
        y = x * tl.sigmoid(x)  # SiLU = x * sigmoid(x)
        tl.store(y_ptr + offs, y, mask=mask)

    def triton_silu(x):
        y = torch.empty_like(x)
        n = x.numel()
        BLOCK = 1024
        grid = (triton.cdiv(n, BLOCK),)
        elementwise_silu_kernel[grid](x, y, n, BLOCK=BLOCK)
        return y

    if torch.cuda.is_available():
        x = torch.randn(1024*1024, device='cuda', dtype=torch.float32)
        y_triton = triton_silu(x)
        y_torch = F.silu(x)
        print(f'Triton SiLU correct: {torch.allclose(y_triton, y_torch, atol=1e-4)}')
    else:
        print('Triton requires CUDA GPU')
        print('Triton kernel syntax demo:')
        print('  @triton.jit')
        print('  def kernel(x_ptr, y_ptr, n, BLOCK: tl.constexpr):')
        print('      pid = tl.program_id(0)')
        print('      offs = pid * BLOCK + tl.arange(0, BLOCK)')
        print('      x = tl.load(x_ptr + offs)')
        print('      tl.store(y_ptr + offs, x * tl.sigmoid(x))')
except ImportError:
    print('pip install triton for GPU kernel development')
    print('Triton compiles Python DSL to PTX/CUBIN for GPU execution')


## 2. Benchmarking Custom vs PyTorch Kernels

The key question: is the custom Triton kernel faster than PyTorch's built-in? For elementwise ops, they should be similar (both memory-bandwidth-limited).


In [ ]:
print('Kernel bandwidth benchmark:')
import time

def bench_fn(fn, x, warmup=10, iters=100):
    for _ in range(warmup): fn(x)
    if x.device.type=='cuda': torch.cuda.synchronize()
    t0=time.perf_counter()
    for _ in range(iters): fn(x)
    if x.device.type=='cuda': torch.cuda.synchronize()
    return (time.perf_counter()-t0)/iters*1e6  # us

device = 'cuda' if torch.cuda.is_available() else 'cpu'
x = torch.randn(1024*1024, device=device, dtype=torch.float32)

t_torch = bench_fn(F.silu, x)
print(f'PyTorch SiLU: {t_torch:.1f} us')
bytes_moved = x.numel() * 4 * 2  # read + write, FP32
bw_gb_s = bytes_moved / (t_torch / 1e6) / 1e9
print(f'Bandwidth: {bw_gb_s:.1f} GB/s')
print(f'(DGX Spark spec: ~273 GB/s, PyTorch overhead typically 5-30%)')


## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- Triton is a Python DSL for writing GPU kernels at the tile level.
- Triton's tile-level abstraction means you never write explicit thread indices — you write programs over tiles, and the compiler maps tiles to warps/blocks automatically..
- Day 42 implementation complete.
